# Hotel Cancellation — Feature Prep

Loads the processed train/test splits from `end_to_end.ipynb`, separates features (`X`) from the target (`Y`), carves a stratified validation split out of train, scales `X`'s numeric columns (fit on train only), then saves the resulting `X`/`Y` train/val/test CSVs for use by downstream models.

## 1. Setup

In [18]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = "../data/processed"

## 2. Load Processed Data

In [19]:
train_df = pd.read_csv(f"{PROCESSED_DIR}/processed_data_train.csv")
test_df = pd.read_csv(f"{PROCESSED_DIR}/processed_data_test.csv")

print(f"Train shape: {train_df.shape}   Test shape: {test_df.shape}")
train_df.head(5)

Train shape: (95368, 45)   Test shape: (23841, 45)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,arrival_date_month_December,arrival_date_month_February,arrival_date_month_January,arrival_date_month_July,arrival_date_month_June,arrival_date_month_March,arrival_date_month_May,arrival_date_month_November,arrival_date_month_October,arrival_date_month_September
0,0,0,11,2017,6,10,0,2,2,0,...,0,1,0,0,0,0,0,0,0,0
1,1,1,128,2016,28,7,0,3,2,0,...,0,0,0,1,0,0,0,0,0,0
2,1,0,7,2016,51,14,0,2,1,0,...,1,0,0,0,0,0,0,0,0,0
3,0,1,77,2015,41,8,0,3,2,2,...,0,0,0,0,0,0,0,0,1,0
4,0,0,6,2016,29,13,0,1,1,0,...,0,0,0,1,0,0,0,0,0,0


## 3. Split into X / Y

`lead_time` and `adr` are dropped in favor of their log-transformed counterparts (`lead_time_log`, `adr_log`), matching the redundancy call made in the EDA notebook.

In [20]:
drop_cols = ["is_canceled", "lead_time", "adr"]

Y_train = train_df[["is_canceled"]]
Y_test = test_df[["is_canceled"]]


X_train = train_df.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)

print(f"X_train shape: {X_train.shape}   Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}   Y_test shape: {Y_test.shape}")

X_train shape: (95368, 42)   Y_train shape: (95368, 1)
X_test shape: (23841, 42)   Y_test shape: (23841, 1)


### Drop rows with missing values

Any row with a `NaN` in its features or target is dropped (train and test independently), rather than imputed.

In [21]:
train_mask = X_train.notna().all(axis=1) & Y_train.notna().all(axis=1)
test_mask = X_test.notna().all(axis=1) & Y_test.notna().all(axis=1)

X_train, Y_train = X_train[train_mask], Y_train[train_mask]
X_test, Y_test = X_test[test_mask], Y_test[test_mask]

print(f"Dropped {(~train_mask).sum()} NaN rows from train, {(~test_mask).sum()} from test")
print(f"X_train shape: {X_train.shape}   Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}   Y_test shape: {Y_test.shape}")

Dropped 0 NaN rows from train, 1 from test
X_train shape: (95368, 42)   Y_train shape: (95368, 1)
X_test shape: (23840, 42)   Y_test shape: (23840, 1)


## 4. Scale Numeric Features & Save

`StandardScaler` is fit on train only and applied to train, val, and test. Only genuinely continuous/count columns are scaled — binary and one-hot columns (e.g. `hotel`, `deposit_type_*`, `arrival_date_month_*`) are left as 0/1, since standardizing a column with tiny variance (a rare one-hot category) blows up its scaled values. `X`/`Y` for all three splits are saved to CSV after scaling.

In [22]:
binary_cols = [c for c in X_train.columns if X_train[c].nunique() <= 2]
numeric_cols = [c for c in X_train.columns if c not in binary_cols]

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(f"Binary/one-hot columns left unscaled: {len(binary_cols)}")
print(f"Numeric columns scaled: {len(numeric_cols)}")

X_train.to_csv(f"{PROCESSED_DIR}/X_train.csv", index=False)
X_test.to_csv(f"{PROCESSED_DIR}/X_test.csv", index=False)
Y_train.to_csv(f"{PROCESSED_DIR}/Y_train.csv", index=False)
Y_test.to_csv(f"{PROCESSED_DIR}/Y_test.csv", index=False)
print("Saved X/Y train, val, test CSVs (X scaled)")

Binary/one-hot columns left unscaled: 23
Numeric columns scaled: 19
Saved X/Y train, val, test CSVs (X scaled)
